# Memory-Augmented Chatbot with Knowledge Graph and Hybrid RAG System

This notebook is a **systematic, runnable implementation** of the project described in
`Memory_Augmented_Chatbot_Problem_Statement.docx`. It follows the exact methodology from the
problem statement, step by step:

1. Data Pipeline (scrape → clean → chunk)
2. Embedding & Vector Storage (FAISS)
3. Knowledge Graph Construction (entities + relationships, NetworkX — optional Neo4j)
4. RAG Pipeline (retrieval + generation)
5. LangGraph Workflow (Model / Memory / RAG / Tool nodes + routing)
6. Dynamic Tool Integration (live/real-time data)
7. Evaluation Framework (context relevance, answer correctness, faithfulness)

**How to use this notebook**
- Run cells top to bottom in Google Colab (`Runtime → Run all` also works after you add your API key).
- You only *need* an OpenAI API key (for the LLM). Everything else (embeddings, vector store,
  knowledge graph, memory) runs locally/free inside Colab.
- Neo4j is optional — a free local graph (NetworkX) is used by default so the notebook runs with
  zero external services. A commented-out Neo4j Aura integration is included if you want the
  "real" graph database from the tech stack.

> Swap `ChatOpenAI` for any other LangChain chat model (Groq, Gemini, Anthropic, etc.) in the
> `get_llm()` function in Section 0 if you don't have an OpenAI key — the rest of the notebook is
> provider-agnostic.


## 0. Setup — Install Dependencies

In [ ]:
# Core LLM orchestration
!pip -q install langgraph langchain langchain-openai langchain-community

# Embeddings + vector store
!pip -q install sentence-transformers faiss-cpu

# Knowledge graph
!pip -q install networkx pyvis

# Entity/relationship extraction
!pip -q install spacy
!python -m spacy download en_core_web_sm -q

# Web scraping
!pip -q install beautifulsoup4 requests lxml

# Dynamic/real-time tool (free, no API key needed)
!pip -q install duckduckgo-search

# Optional: Neo4j driver (only needed if you enable the Neo4j section below)
!pip -q install neo4j

print("All dependencies installed.")


In [ ]:
import os
import json
import time
import getpass
import requests
import numpy as np
import networkx as nx
from bs4 import BeautifulSoup
from typing import TypedDict, List, Dict, Any, Optional

# ---- API KEYS -------------------------------------------------------------
# You will be prompted once; the key is kept only in this Colab session.
if "OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"]:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

print("Environment ready.")


In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

def get_llm(temperature: float = 0.2):
    """Central place to swap the LLM provider.
    Replace this with ChatGroq / ChatGoogleGenerativeAI / ChatAnthropic etc.
    if you don't want to use OpenAI — the rest of the notebook only calls get_llm().
    """
    return ChatOpenAI(model="gpt-4o-mini", temperature=temperature)

def get_embeddings():
    """Embedding model used for the vector store."""
    return OpenAIEmbeddings(model="text-embedding-3-small")

llm = get_llm()
embeddings = get_embeddings()
print("LLM and embedding clients initialized.")


## 1. Data Pipeline (Static Knowledge Layer)

**Step 1 of the methodology:** web scraping → cleaning → chunking.

Replace `SOURCE_URLS` with the pages you want your chatbot to know about (docs, wiki pages,
blog posts, company FAQs, etc.).


In [ ]:
SOURCE_URLS = [
    "https://en.wikipedia.org/wiki/Retrieval-augmented_generation",
    "https://en.wikipedia.org/wiki/Knowledge_graph",
    "https://en.wikipedia.org/wiki/Chatbot",
]

def scrape_page(url: str) -> str:
    """Fetch a page and extract clean visible text."""
    try:
        resp = requests.get(url, timeout=15, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"Failed to fetch {url}: {e}")
        return ""

    soup = BeautifulSoup(resp.text, "lxml")

    # Strip non-content elements
    for tag in soup(["script", "style", "nav", "footer", "header", "aside", "table"]):
        tag.decompose()

    paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
    text = "\n".join(p for p in paragraphs if len(p) > 40)
    return text

def clean_text(text: str) -> str:
    """Basic normalization/cleaning."""
    text = text.replace("\xa0", " ")
    text = " ".join(text.split())  # collapse whitespace
    return text

raw_documents = []
for url in SOURCE_URLS:
    print(f"Scraping: {url}")
    text = scrape_page(url)
    if text:
        raw_documents.append({"source": url, "text": clean_text(text)})
    time.sleep(1)  # be polite to servers

print(f"\nScraped {len(raw_documents)} documents.")
if raw_documents:
    print("Sample (first 400 chars):\n", raw_documents[0]["text"][:400])


In [ ]:
def chunk_text(text: str, chunk_size: int = 800, overlap: int = 100) -> List[str]:
    """Simple sliding-window chunker (character-based, dependency-free)."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return [c.strip() for c in chunks if len(c.strip()) > 50]

all_chunks = []  # list of dicts: {"text":..., "source":...}
for doc in raw_documents:
    for chunk in chunk_text(doc["text"]):
        all_chunks.append({"text": chunk, "source": doc["source"]})

print(f"Created {len(all_chunks)} chunks from {len(raw_documents)} documents.")


## 2. Embedding & Vector Storage (FAISS)

**Step 2 of the methodology:** generate embeddings, store in a vector database. FAISS is used
here (swap for Chroma with `langchain_community.vectorstores.Chroma` if you prefer a persistent
on-disk store).


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain.docstore.document import Document

lc_documents = [
    Document(page_content=c["text"], metadata={"source": c["source"]})
    for c in all_chunks
]

vector_store = FAISS.from_documents(lc_documents, embeddings)
print(f"FAISS vector store built with {len(lc_documents)} chunks.")

# Persist to disk so you don't have to re-embed every run
vector_store.save_local("faiss_index")
print("Saved index to ./faiss_index")

# To reload later:
# vector_store = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)


## 3. Knowledge Graph Construction

**Step 3 of the methodology:** entity extraction → relationship mapping → graph storage.

We use spaCy for lightweight entity extraction and co-occurrence-based relationships, stored in
an in-memory **NetworkX** graph (free, no server needed). A commented **Neo4j** block is provided
below if you have a Neo4j Aura free-tier instance and want the "real" graph database.


In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")
kg = nx.DiGraph()

def extract_entities_relations(text: str, source: str):
    """Extract named entities and connect entities that co-occur in the same sentence
    (a simple, fast relationship heuristic that works without a dedicated RE model)."""
    doc = nlp(text)
    for sent in doc.sents:
        ents = [ent for ent in sent.ents if ent.label_ in
                {"PERSON", "ORG", "GPE", "PRODUCT", "EVENT", "WORK_OF_ART", "NORP", "FAC", "LAW"}]
        for ent in ents:
            kg.add_node(ent.text, label=ent.label_)
        for i in range(len(ents)):
            for j in range(i + 1, len(ents)):
                kg.add_edge(ents[i].text, ents[j].text, relation="co_occurs_with", source=source)

for doc in raw_documents:
    extract_entities_relations(doc["text"], doc["source"])

print(f"Knowledge graph built: {kg.number_of_nodes()} entities, {kg.number_of_edges()} relations.")
print("Sample nodes:", list(kg.nodes)[:10])


In [ ]:
# --- Optional: visualize the knowledge graph inline ---
from pyvis.network import Network

net = Network(notebook=True, cdn_resources="in_line", height="500px", width="100%", directed=True)
# Limit to top-connected nodes so the graph stays readable
top_nodes = sorted(kg.degree, key=lambda x: x[1], reverse=True)[:40]
sub_kg = kg.subgraph([n for n, _ in top_nodes])
net.from_nx(sub_kg)
net.show("knowledge_graph.html")

from IPython.display import HTML
HTML("knowledge_graph.html")


In [ ]:
# --- Optional: push the graph to Neo4j instead of / in addition to NetworkX ---
# Uncomment and fill in your Neo4j Aura credentials (https://neo4j.com/cloud/aura-free/) to use
# a real graph database as specified in the tech stack.

# from neo4j import GraphDatabase
#
# NEO4J_URI = "neo4j+s://<your-instance>.databases.neo4j.io"
# NEO4J_USER = "neo4j"
# NEO4J_PASSWORD = getpass.getpass("Neo4j password: ")
#
# driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
#
# def push_to_neo4j(graph: nx.DiGraph):
#     with driver.session() as session:
#         for node, data in graph.nodes(data=True):
#             session.run(
#                 "MERGE (e:Entity {name: $name}) SET e.label = $label",
#                 name=node, label=data.get("label", "UNKNOWN"),
#             )
#         for u, v, data in graph.edges(data=True):
#             session.run(
#                 """MATCH (a:Entity {name: $u}), (b:Entity {name: $v})
#                    MERGE (a)-[r:RELATED {type: $rel}]->(b)""",
#                 u=u, v=v, rel=data.get("relation", "related_to"),
#             )
#
# push_to_neo4j(kg)
# print("Graph pushed to Neo4j.")


In [ ]:
def query_knowledge_graph(entity: str, depth: int = 1) -> List[str]:
    """Return a small set of facts about an entity from the knowledge graph."""
    facts = []
    matches = [n for n in kg.nodes if entity.lower() in n.lower()]
    for m in matches:
        for neighbor in kg.successors(m):
            rel = kg[m][neighbor].get("relation", "related_to")
            facts.append(f"{m} --{rel}--> {neighbor}")
        for neighbor in kg.predecessors(m):
            rel = kg[neighbor][m].get("relation", "related_to")
            facts.append(f"{neighbor} --{rel}--> {m}")
    return facts[:10]

# quick smoke test
print(query_knowledge_graph("Google") or "No matches — try an entity that appears in your sources.")


## 4. RAG Pipeline

**Step 4 of the methodology:** query embedding → similarity search → context retrieval → LLM
answer generation.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant. Answer the user's question using ONLY the provided context "
     "and knowledge graph facts. If the answer is not contained in them, say you don't know. "
     "Cite the source when relevant."),
    ("human",
     "Context:\n{context}\n\nKnowledge Graph Facts:\n{kg_facts}\n\nQuestion: {question}"),
])

def retrieve_context(query: str, k: int = 4):
    docs = vector_store.similarity_search(query, k=k)
    context = "\n\n".join(f"[{d.metadata.get('source')}] {d.page_content}" for d in docs)
    return context, docs

def rag_answer(query: str) -> Dict[str, Any]:
    context, docs = retrieve_context(query)
    kg_facts = "\n".join(query_knowledge_graph(query)) or "No relevant graph facts found."
    chain = RAG_PROMPT | llm
    response = chain.invoke({"context": context, "kg_facts": kg_facts, "question": query})
    return {
        "answer": response.content,
        "context": context,
        "kg_facts": kg_facts,
        "sources": list({d.metadata.get("source") for d in docs}),
    }

# quick smoke test
result = rag_answer("What is retrieval-augmented generation?")
print(result["answer"])
print("\nSources:", result["sources"])


## 5. Long-Term Memory

Per-user memory that persists across turns (and across sessions, since it's saved to disk).
Memory is itself embedded so it can be *semantically* retrieved — e.g. "what did I say my
favorite language was?" — not just replayed in order.


In [ ]:
MEMORY_PATH = "user_memory.json"

def load_memory() -> Dict[str, List[Dict[str, str]]]:
    if os.path.exists(MEMORY_PATH):
        with open(MEMORY_PATH, "r") as f:
            return json.load(f)
    return {}

def save_memory(memory: Dict[str, List[Dict[str, str]]]):
    with open(MEMORY_PATH, "w") as f:
        json.dump(memory, f, indent=2)

memory_store = load_memory()

def add_memory(user_id: str, role: str, content: str):
    memory_store.setdefault(user_id, [])
    memory_store[user_id].append({"role": role, "content": content, "ts": time.time()})
    save_memory(memory_store)

def get_recent_memory(user_id: str, n: int = 6) -> str:
    turns = memory_store.get(user_id, [])[-n:]
    return "\n".join(f"{t['role']}: {t['content']}" for t in turns)

def summarize_user_preferences(user_id: str) -> str:
    """Ask the LLM to distill durable facts/preferences from the conversation history —
    this is what gives the bot 'long-term' memory instead of just a chat log."""
    history = "\n".join(f"{t['role']}: {t['content']}" for t in memory_store.get(user_id, []))
    if not history:
        return "No known preferences yet."
    prompt = (
        "From the conversation history below, extract durable user facts/preferences "
        "(likes, role, goals, constraints). Return a short bullet list. If none, say 'None'.\n\n"
        f"{history}"
    )
    return llm.invoke(prompt).content

print("Memory system ready. Existing users:", list(memory_store.keys()))


## 6. Dynamic Tool Integration

**Step 6 of the methodology:** integrate APIs / live data so the bot can answer questions that
static knowledge (scraped pages) can't — anything time-sensitive or not in the corpus.

Uses DuckDuckGo search (free, no API key) as the live-data tool.


In [ ]:
from duckduckgo_search import DDGS

def web_search_tool(query: str, max_results: int = 4) -> str:
    """Fetches live web results for time-sensitive / out-of-corpus questions."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
        if not results:
            return "No live results found."
        return "\n\n".join(f"{r['title']}: {r['body']} ({r['href']})" for r in results)
    except Exception as e:
        return f"Tool error: {e}"

# quick smoke test
print(web_search_tool("current weather in Bhubaneswar")[:500])


## 7. LangGraph Workflow — Orchestration Layer

**Step 5 of the methodology.** This is the "Dynamic Intelligence Layer" from the system
overview: a router node decides whether a query needs the **RAG** node, the **Tool** node, or
can be answered directly from **Memory**, then a final **Model** node composes the reply using
whatever context was gathered.

Graph: `router → {rag_node | tool_node | memory_node} → responder`


In [ ]:
from langgraph.graph import StateGraph, END

class ChatState(TypedDict):
    user_id: str
    question: str
    route: str
    rag_result: Optional[Dict[str, Any]]
    tool_result: Optional[str]
    memory_context: str
    final_answer: str

ROUTER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Classify the user question into exactly one category:\n"
     "- 'rag' — general knowledge questions answerable from a static knowledge base\n"
     "- 'tool' — needs real-time / current / live information (news, weather, prices, 'today', 'latest')\n"
     "- 'memory' — asks about the user themselves or prior conversation ('what did I tell you', 'my name')\n"
     "Respond with ONLY one word: rag, tool, or memory."),
    ("human", "{question}"),
])

def router_node(state: ChatState) -> ChatState:
    route = (ROUTER_PROMPT | llm).invoke({"question": state["question"]}).content.strip().lower()
    if route not in {"rag", "tool", "memory"}:
        route = "rag"
    state["route"] = route
    return state

def rag_node(state: ChatState) -> ChatState:
    state["rag_result"] = rag_answer(state["question"])
    return state

def tool_node(state: ChatState) -> ChatState:
    state["tool_result"] = web_search_tool(state["question"])
    return state

def memory_node(state: ChatState) -> ChatState:
    state["memory_context"] = get_recent_memory(state["user_id"])
    return state

RESPONDER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a personalized, memory-aware assistant. Use whatever context is provided "
     "(retrieved knowledge, live tool results, and/or conversation memory) to answer clearly "
     "and concisely. Do not mention internal routing or tool names."),
    ("human",
     "Recent conversation memory:\n{memory_context}\n\n"
     "Retrieved knowledge (if any):\n{rag_context}\n\n"
     "Live tool results (if any):\n{tool_context}\n\n"
     "Question: {question}"),
])

def responder_node(state: ChatState) -> ChatState:
    rag_context = state["rag_result"]["answer"] if state.get("rag_result") else "N/A"
    tool_context = state.get("tool_result") or "N/A"
    memory_context = state.get("memory_context") or get_recent_memory(state["user_id"])
    response = (RESPONDER_PROMPT | llm).invoke({
        "memory_context": memory_context,
        "rag_context": rag_context,
        "tool_context": tool_context,
        "question": state["question"],
    })
    state["final_answer"] = response.content
    return state

def route_decision(state: ChatState) -> str:
    return state["route"]

workflow = StateGraph(ChatState)
workflow.add_node("router", router_node)
workflow.add_node("rag", rag_node)
workflow.add_node("tool", tool_node)
workflow.add_node("memory", memory_node)
workflow.add_node("responder", responder_node)

workflow.set_entry_point("router")
workflow.add_conditional_edges("router", route_decision, {
    "rag": "rag", "tool": "tool", "memory": "memory",
})
workflow.add_edge("rag", "responder")
workflow.add_edge("tool", "responder")
workflow.add_edge("memory", "responder")
workflow.add_edge("responder", END)

chatbot_graph = workflow.compile()
print("LangGraph workflow compiled.")


In [ ]:
# --- Visualize the graph structure ---
from IPython.display import Image, display
try:
    display(Image(chatbot_graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Mermaid rendering unavailable in this environment:", e)
    print(chatbot_graph.get_graph().draw_ascii())


## 8. Chat Function — Putting It All Together

This wraps the graph with memory read/write, so every turn is stored and future turns can draw
on it.


In [ ]:
def chat(user_id: str, question: str, verbose: bool = True) -> str:
    state: ChatState = {
        "user_id": user_id,
        "question": question,
        "route": "",
        "rag_result": None,
        "tool_result": None,
        "memory_context": get_recent_memory(user_id),
        "final_answer": "",
    }
    result = chatbot_graph.invoke(state)

    add_memory(user_id, "user", question)
    add_memory(user_id, "assistant", result["final_answer"])

    if verbose:
        print(f"[routed to: {result['route']}]")
    return result["final_answer"]

# --- Demo conversation ---
user = "demo_user"
print("Bot:", chat(user, "Hi, my name is Alex and I'm interested in AI systems."))
print()
print("Bot:", chat(user, "What is retrieval-augmented generation?"))
print()
print("Bot:", chat(user, "What's the latest news about OpenAI?"))
print()
print("Bot:", chat(user, "What's my name and what am I interested in?"))


## 9. Evaluation Framework

**Step 7 of the methodology.** Since this project doesn't rely on a labeled test set, we use an
**LLM-as-judge** approach (the standard technique when ground truth is unavailable) to score:

- **Context Relevance** — was the retrieved context actually relevant to the question?
- **Faithfulness** — is the answer grounded in the retrieved context (not hallucinated)?
- **Answer Correctness** — does the answer actually address the question well?

Swap in [RAGAS](https://github.com/explodinggraphs/ragas) for a more rigorous, standardized
version of the same idea if you want published metrics.


In [ ]:
EVAL_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are an evaluation judge for a RAG system. Score the following on a 1-5 scale each:\n"
     "- context_relevance: how relevant the retrieved context is to the question\n"
     "- faithfulness: how well the answer is grounded in the context (no hallucination)\n"
     "- answer_correctness: how well the answer addresses the question\n"
     "Return ONLY valid JSON: {{\"context_relevance\": int, \"faithfulness\": int, "
     "\"answer_correctness\": int, \"justification\": str}}"),
    ("human",
     "Question: {question}\n\nContext:\n{context}\n\nAnswer:\n{answer}"),
])

def evaluate_response(question: str, context: str, answer: str) -> Dict[str, Any]:
    raw = (EVAL_PROMPT | llm).invoke({
        "question": question, "context": context, "answer": answer,
    }).content
    try:
        cleaned = raw.strip().strip("```").replace("json", "", 1).strip()
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"raw_response": raw}

# --- Run evaluation over a small test set ---
test_questions = [
    "What is retrieval-augmented generation?",
    "What is a knowledge graph used for?",
    "How do chatbots work?",
]

eval_results = []
for q in test_questions:
    result = rag_answer(q)
    score = evaluate_response(q, result["context"], result["answer"])
    eval_results.append({"question": q, **score})
    print(f"Q: {q}\n  -> {score}\n")

import pandas as pd
eval_df = pd.DataFrame(eval_results)
eval_df


In [ ]:
# Aggregate scores
numeric_cols = [c for c in ["context_relevance", "faithfulness", "answer_correctness"] if c in eval_df.columns]
if numeric_cols:
    print("Average scores across test set:")
    print(eval_df[numeric_cols].mean())


## 10. Next Steps

- **Persistence:** swap the JSON memory store for MongoDB/PostgreSQL (as in the tech stack) for
  production use.
- **Scale the corpus:** add more `SOURCE_URLS`, or swap `scrape_page` for a Scrapy crawler for
  large-scale ingestion.
- **Serve it:** wrap `chat()` in a FastAPI endpoint (`POST /chat {user_id, question}`) to match
  the tech stack's API framework.
- **Harden routing:** replace the single-word LLM router with a LangGraph `tool-calling` router
  or a fine-tuned classifier if you need higher routing accuracy.
- **Neo4j:** enable the commented Neo4j block in Section 3 for a persistent, queryable graph
  database instead of the in-memory NetworkX graph.
